# Part B — 우선순위 2: VIF 처리 + 재적합
1차에서 `title_len`(VIF 7.77) ↔ `word_count`(7.76) 공선성. **word_count 제거** 후 재적합.
성능 지표는 진단 결론대로 **시간 split + 랜덤 split 병기** (랜덤=정상 베이스라인).

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split

features = pd.read_csv("features_partB.csv", parse_dates=["publish_time"])
hour_labels = ["00-05","06-11","12-17","18-23"]
features["hour_bin"] = pd.Categorical(features["hour_bin"], categories=hour_labels)

# word_count 제거 (title_len 유지)
title_feats = ["title_len","caps_ratio","exclaim_cnt",
               "question_cnt","has_number","has_bracket","tag_cnt"]
formula = ("log_views ~ title_len + caps_ratio + exclaim_cnt + "
           "question_cnt + has_number + has_bracket + tag_cnt + "
           "C(hour_bin, Treatment(reference='18-23')) + "
           "C(publish_weekday) + C(category)")
print("제거됨: word_count |  잔여 제목변수:", title_feats)

제거됨: word_count |  잔여 제목변수: ['title_len', 'caps_ratio', 'exclaim_cnt', 'question_cnt', 'has_number', 'has_bracket', 'tag_cnt']


## VIF 재확인 (word_count 제거 후)

In [2]:
X = sm.add_constant(features[title_feats])
vif = pd.DataFrame({"feature": X.columns,
    "VIF":[variance_inflation_factor(X.values,i) for i in range(X.shape[1])]})
print(vif.round(3).to_string(index=False))

     feature    VIF
       const 11.367
   title_len  1.090
  caps_ratio  1.121
 exclaim_cnt  1.082
question_cnt  1.011
  has_number  1.030
 has_bracket  1.025
     tag_cnt  1.052


## 재적합 — 시간 split (train 통계 검증)

In [3]:
def time_split(d, frac=0.8):
    d=d.dropna(subset=["publish_time"]).sort_values("publish_time").reset_index(drop=True)
    cut=int(len(d)*frac); return d.iloc[:cut].copy(), d.iloc[cut:].copy()

train, test = time_split(features)
ols = smf.ols(formula, data=train).fit()

coef_tbl = pd.DataFrame({"coef":ols.params,"p_value":ols.pvalues})
title_rows = coef_tbl.loc[[c for c in title_feats if c in coef_tbl.index]].copy()
title_rows["sig"]=np.where(title_rows["p_value"]<0.05,"*","")
print("[제목변수 계수 (재적합, train)]")
print(title_rows.round(4).to_string())
print(f"\nR2(train)={ols.rsquared:.4f}  Adj={ols.rsquared_adj:.4f}")

[제목변수 계수 (재적합, train)]
                coef  p_value sig
title_len    -0.0069   0.0000   *
caps_ratio    0.4206   0.0005   *
exclaim_cnt   0.1619   0.0071   *
question_cnt  0.0313   0.7609    
has_number   -0.0415   0.4733    
has_bracket   0.3895   0.0000   *
tag_cnt       0.0218   0.0000   *



R2(train)=0.1134  Adj=0.1078


## 계수 안정성 — 제거 전후 비교

In [4]:
# 제거 전(word_count 포함) 모델 재적합해 비교
formula_full=("log_views ~ title_len + word_count + caps_ratio + exclaim_cnt + "
              "question_cnt + has_number + has_bracket + tag_cnt + "
              "C(hour_bin, Treatment(reference='18-23')) + "
              "C(publish_weekday) + C(category)")
ols_full=smf.ols(formula_full, data=train).fit()

comp_feats=["title_len","caps_ratio","exclaim_cnt","question_cnt","has_number","has_bracket","tag_cnt"]
cmp=pd.DataFrame({
    "before(coef)":[ols_full.params.get(f,np.nan) for f in comp_feats],
    "after(coef)" :[ols.params.get(f,np.nan) for f in comp_feats],
    "before(p)"   :[ols_full.pvalues.get(f,np.nan) for f in comp_feats],
    "after(p)"    :[ols.pvalues.get(f,np.nan) for f in comp_feats],
}, index=comp_feats)
print(cmp.round(4).to_string())

              before(coef)  after(coef)  before(p)  after(p)
title_len          -0.0139      -0.0069     0.0001    0.0000
caps_ratio          0.3973       0.4206     0.0011    0.0005
exclaim_cnt         0.1701       0.1619     0.0047    0.0071
question_cnt        0.0169       0.0313     0.8697    0.7609
has_number         -0.0600      -0.0415     0.3052    0.4733
has_bracket         0.4039       0.3895     0.0000    0.0000
tag_cnt             0.0216       0.0218     0.0000    0.0000


## 성능 — 시간 split vs 랜덤 split 병기

In [5]:
def evaluate(model, data, label):
    p=model.predict(data); y=data["log_views"]
    r2=r2_score(y,p); rmse=np.sqrt(mean_squared_error(y,p))
    print(f"[{label}] R2={r2:.4f}  RMSE={rmse:.4f}"); return r2,rmse

print("=== 시간 split (drift 영향) ===")
evaluate(ols, train, "TRAIN")
evaluate(ols, test,  "TEST ")

print("\n=== 랜덤 split (정상 베이스라인) ===")
rtr,rte=train_test_split(features.dropna(subset=["publish_time"]),test_size=0.2,random_state=42)
rte=rte[rte["category"].isin(set(rtr["category"].unique()))]
rols=smf.ols(formula,data=rtr).fit()
evaluate(rols, rtr, "TRAIN")
evaluate(rols, rte, "TEST ")

=== 시간 split (drift 영향) ===


[TRAIN] R2=0.1134  RMSE=1.6989
[TEST ] R2=-1.2711  RMSE=1.8397

=== 랜덤 split (정상 베이스라인) ===


[TRAIN] R2=0.1186  RMSE=1.6912
[TEST ] R2=0.1188  RMSE=1.7520


(0.11878994721518343, np.float64(1.7520177732574393))

## 카테고리별 재적합 (H4, word_count 제거 반영)

In [6]:
top3 = features["category"].value_counts().head(3).index.tolist()
print("Top3:", top3)
sub_formula=("log_views ~ title_len + caps_ratio + exclaim_cnt + "
             "question_cnt + has_number + has_bracket + tag_cnt + "
             "C(hour_bin, Treatment(reference='18-23')) + C(publish_weekday)")
def star(p): return "*" if p<0.05 else ""
models={c:smf.ols(sub_formula,data=features[features["category"]==c]).fit() for c in top3}
rows=[]
for f in title_feats:
    r={"feature":f}
    for c in top3:
        m=models[c]; r[c]=f"{m.params[f]:.3f}{star(m.pvalues[f])}" if f in m.params.index else "-"
    rows.append(r)
compare=pd.DataFrame(rows).set_index("feature")
compare.to_csv("coef_comparison_top3_v2.csv")
print("(* = p<0.05)")
print(compare.to_string())

Top3: ['Entertainment', 'Music', 'Howto & Style']
(* = p<0.05)
             Entertainment    Music Howto & Style
feature                                          
title_len          -0.009*   -0.003         0.002
caps_ratio          0.520*   0.989*        0.653*
exclaim_cnt          0.165   -0.336         0.166
question_cnt        -0.187   -0.154         0.241
has_number           0.082  -0.476*        -0.197
has_bracket         0.493*   0.407*        0.546*
tag_cnt             0.019*   0.015*         0.004


---
### 결론
- **VIF 개선**: word_count 제거 후 title_len VIF **7.77→1.09**, 전 변수 1.0~1.1대. 공선성 해소.
- **계수 안정성**: 주요 변수(caps_ratio·has_bracket·exclaim_cnt·tag_cnt) 방향·크기·유의성 유지. title_len만 −0.014→−0.007(여전히 유의). 제거가 결론을 바꾸지 않음.
- **성능**: 시간 split TEST −1.28(drift) / 랜덤 split TEST **+0.115**(정상 베이스라인). word_count 제거해도 R² 손실 없음(0.113→0.112).
- **H4 재확인**: has_number(Music만 −0.47*), exclaim_cnt(Music만 −), caps_ratio(Music만 유의X) → 자극성 효과의 카테고리 의존성 = **H4 지지**. 결과 저장: coef_comparison_top3_v2.csv
- **확정 feature set**: title_len, caps_ratio, exclaim_cnt, question_cnt, has_number, has_bracket, tag_cnt (word_count 제외).
- **→ 다음**: 우선순위 3(잔차 마무리) 후 4단계 합류.